In [ ]:
from src.feature_engineering import feature_engineering

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import re


In [ ]:
# Load engineered data
X_train, X_val, X_test, y_train, y_val, y_test, scaler = feature_engineering()

## Feature selection

In [ ]:

# Group one-hot columns by their base category
groups = {}
for col in X_train.columns:
    base = re.sub(r'_[^_]+$', '', col)  # removes the last suffix after underscore
    groups.setdefault(base, []).append(col)

# Initial feature importance selection
selector_model = RandomForestClassifier(n_estimators=100, random_state=42)
selector_model.fit(X_train, y_train)

importances = pd.Series(selector_model.feature_importances_, index=X_train.columns)
top_features = importances.nlargest(25).index

# Enforce full group inclusion
final_features = set()
for g, cols in groups.items():
    if any(c in top_features for c in cols):
        final_features.update(cols)

X_train = X_train[list(final_features)]
X_val = X_val[list(final_features)]
X_test = X_test[list(final_features)]
print("Final grouped features:", final_features)


In [ ]:
for df in [X_train, X_val, X_test]:
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]

# Set MLflow tracking URI and experiment
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("model-comparison")

# Define models with their configurations
models = {
    "Logistic_Regression": {
        "model": LogisticRegression(),
        "log_function": mlflow.sklearn.log_model
    },
    "XGBoost": {
        "model": XGBClassifier(),
        "log_function": mlflow.xgboost.log_model
    },
    "Decision_Tree": {
        "model": DecisionTreeClassifier(random_state=0),
        "log_function": mlflow.sklearn.log_model
    },
    "Random_Forest": {
        "model": RandomForestClassifier(class_weight='balanced'),
        "log_function": mlflow.sklearn.log_model
    }
}

# Dictionary to store results
results = {}

# Train and track each model
for model_name, model_config in models.items():
    print(f"\nTraining {model_name}...")

    with mlflow.start_run(run_name=model_name):
        # Get model and logging function
        model = model_config["model"]
        log_function = model_config["log_function"]

        # Log model parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("val_size", len(X_val))

        # Log specific model parameters
        for param_name, param_value in model.get_params().items():
            mlflow.log_param(param_name, param_value)

        # Train model
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_val)

        # Calculate metrics
        accuracy = accuracy_score(y_val, predictions)
        f1 = f1_score(y_val, predictions, average='weighted')
        precision = precision_score(y_val, predictions, average='weighted')
        recall = recall_score(y_val, predictions, average='weighted')

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # Log the model
        log_function(model, model_name.lower())

        # Store results
        results[model_name] = accuracy

        # Log tags for easy filtering
        mlflow.set_tag("stage", "validation")
        mlflow.set_tag("dataset", "current_dataset")

        print(f"{model_name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}")

# Create results dataframe
df_results = pd.DataFrame({
    "LR": [results["Logistic_Regression"]],
    "XGB": [results["XGBoost"]],
    "DT": [results["Decision_Tree"]],
    "RF": [results["Random_Forest"]]
})

print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(df_results)

# Log the comparison results as an artifact
with mlflow.start_run(run_name="comparison_summary"):
    mlflow.log_param("comparison_type", "all_models")

    # Save and log the results dataframe
    df_results.to_csv("model_comparison.csv", index=False)
    mlflow.log_artifact("model_comparison.csv")

    # Log the best model info
    best_model = max(results, key=results.get)
    best_score = results[best_model]
    mlflow.log_param("best_model", best_model)
    mlflow.log_metric("best_accuracy", best_score)

    print(f"\nBest Model: {best_model} with accuracy: {best_score:.4f}")


In [ ]:
df